# Preprocessing & clustering — SRR11396171 + SRR11396174 (peau humaine, 10x v2)

Matrices kallisto|bustools (`run_kb.sh`). S13_H = peau saine, S16_NL = peau non-lésionnelle (dermatite atopique). Correction de l'effet échantillon avec Harmony.

In [ ]:
# Working directory = data/case2/preprocessing/ (FASTQ, kb_out/, write/ and cache/ live there, not tracked by Git)
from pathlib import Path
import os
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'helpers.R').exists())
os.chdir(root / 'data' / 'case2' / 'preprocessing')


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.io import mmread
from scipy.sparse import csr_matrix

In [ ]:
sc.settings.verbosity = 3
sc.set_figure_params(dpi=80, facecolor="white")
sc.logging.print_header()

In [ ]:
samples = {"SRR11396171": "S13_H", "SRR11396174": "S16_NL"}
results_file = "write/skin_combined.h5ad"

In [ ]:
def read_kb(srr):
    d = f"kb_out/{srr}/counts_filtered"
    X = csr_matrix(mmread(f"{d}/cells_x_genes.mtx")).astype("float32")
    barcodes = pd.read_csv(f"{d}/cells_x_genes.barcodes.txt", header=None)[0].values
    gene_ids = pd.read_csv(f"{d}/cells_x_genes.genes.txt", header=None)[0].values
    names = pd.read_csv(f"{d}/cells_x_genes.genes.names.txt", header=None)[0].fillna(pd.Series(gene_ids)).values
    a = ad.AnnData(X=X, obs=pd.DataFrame(index=barcodes), var=pd.DataFrame({"gene_symbol": names}, index=gene_ids))
    a.obs["srr"] = srr
    a.obs["sample"] = samples[srr]
    return a

adata = ad.concat([read_kb(s) for s in samples], join="outer", index_unique="-", merge="same")
adata.var_names = adata.var["gene_symbol"].astype(str).values
adata.var_names_make_unique()
adata.obs["sample"].value_counts()

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata.obs["sample"].value_counts()

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], groupby="sample", jitter=0.4, multi_panel=True)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", color="sample", show=False, ax=axs[0])
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="sample", show=False, ax=axs[1]);

Seuils QC adaptés à la peau — à ajuster d'après les violins ci-dessus.

In [ ]:
adata = adata[(adata.obs.n_genes_by_counts < 6000) & (adata.obs.pct_counts_mt < 15), :].copy()
adata.layers["counts"] = adata.X.copy()
adata.obs["sample"].value_counts()

In [ ]:
sc.pp.scrublet(adata, batch_key="sample")  # doublets, par échantillon
adata = adata[~adata.obs["predicted_doublet"]].copy()
adata.obs["sample"].value_counts()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=2000, flavor="seurat_v3", batch_key="sample", span=0.5)
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.layers["scaled"] = adata.X.toarray()
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"], layer="scaled")
sc.pp.scale(adata, max_value=10, layer="scaled")

In [ ]:
sc.pp.pca(adata, layer="scaled", svd_solver="arpack", mask_var="highly_variable")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

UMAP avant correction (effet échantillon ?)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["sample", "total_counts", "pct_counts_mt"], wspace=0.4)

Intégration Harmony puis clustering

In [ ]:
sc.external.pp.harmony_integrate(adata, key="sample", basis="X_pca", adjusted_basis="X_pca_harmony")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.7, random_state=0, flavor="igraph", n_iterations=2, directed=False)

In [ ]:
sc.pl.umap(adata, color=["leiden", "sample"], wspace=0.4)
pd.crosstab(adata.obs["leiden"], adata.obs["sample"])

In [ ]:
marker_genes = {
    "Keratinocytes": ["KRT14", "KRT5", "KRT10", "KRT1"],
    "Fibroblasts": ["COL1A1", "DCN", "LUM"],
    "Endothelial": ["PECAM1", "VWF", "ACKR1"],
    "Lymphatic": ["LYVE1", "CCL21"],
    "Pericytes/SMC": ["RGS5", "ACTA2"],
    "T cells": ["CD3D", "PTPRC"],
    "Myeloid": ["LYZ", "CD68", "CD1C"],
    "Mast": ["TPSAB1"],
    "Melanocytes": ["PMEL", "MLANA"],
    "Glandular": ["MUCL1", "KRT18", "SCGB2A2"],
}
marker_genes = {k: [g for g in v if g in adata.var_names] for k, v in marker_genes.items()}
sc.pl.dotplot(adata, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", mask_var="highly_variable", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)
pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)

In [ ]:
adata.write(results_file, compression="gzip")